In [1]:
#Install Python packages for google store scraping inside colab
!pip install google-play-scraper pandas

   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.9 MB ? eta -:--:--
   ------ --------------------------------- 1.6/9.9 MB 5.6 MB/s eta 0:00:02
   --------------- ------------------------ 3.9/9.9 MB 8.2 MB/s eta 0:00:01
   ------------------------- -------------- 6.3/9.9 MB 9.2 MB/s eta 0:00:01
   ---------------------------------- ----- 8.4/9.9 MB 9.8 MB/s eta 0:00:01
   ------------------------------------- -- 9.2/9.9 MB 8.2 MB/s eta 0:00:01
   ---------------------------------------- 9.9/9.9 MB 7.5 MB/s  0:00:01
   ---------------------------------------- 0.0/12.4 MB ? eta -:--:--
   - -------------------------------------- 0.5/12.4 MB 16.0 MB/s eta 0:00:01
   ----- ---------------------------------- 1.8/12.4 MB 4.4 MB/s eta 0:00:03
   ------- -------------------------------- 2.4/12.4 MB 3.6 MB/s eta 0:00:03
   --------- ------------------------------ 2.9/12.4 MB 3.6 MB/s eta 0:00:03
   ------------ --------------


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
#Scrape google store reviews for mobile banking app

from google_play_scraper import reviews, Sort
import pandas as pd

app_id = "ke.co.equitygroup.equitymobile"

result, continuation_token = reviews(
    app_id,
    lang='en',
    country='ke',      # Kenya reviews
    sort=Sort.NEWEST,  # newest
    count=2000
)

df_gp = pd.DataFrame(result)
df_gp.head()

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
0,e2866174-961b-41fd-8d29-209f23c6944a,Grace Nampewo,https://play-lh.googleusercontent.com/a/ACg8oc...,it's a good app,3,0,0.0.392,2026-02-07 08:34:58,Hello Grace! Thank you for rating us. Please s...,2026-02-07 18:33:49,0.0.392
1,c37d17d9-b195-457b-94cf-531f07b3e183,Racheal Nabisubi,https://play-lh.googleusercontent.com/a-/ALV-U...,worst,1,0,0.0.392,2026-02-07 07:04:14,"Hi Rachael, we regret the inconvenience caused...",2026-02-07 18:34:45,0.0.392
2,cc4770c1-e2d9-440d-aae7-638910c0099c,DOB,https://play-lh.googleusercontent.com/a-/ALV-U...,Super and quick though it may work abnormally ...,5,0,0.0.392,2026-02-07 04:28:29,With every new app version we are introducing ...,2026-02-07 18:35:27,0.0.392
3,0ad4e31e-3707-4d1c-9f76-e8b044b254b2,Tracy Disnah,https://play-lh.googleusercontent.com/a-/ALV-U...,not able to send money to others,1,0,NaN,2026-02-07 00:09:00,"Hi Tracy, can you please send us your account ...",2026-02-07 18:35:56,NaN
4,b234477d-8516-4479-ae9d-cb076afe6b50,Lucy Nyambura Wambui.,https://play-lh.googleusercontent.com/a-/ALV-U...,Best app.. makes transactions easier,5,0,0.0.392,2026-02-06 22:08:49,Thank you so much for your feedback.,2026-02-07 18:37:47,0.0.392


In [3]:
#selecting relevant columns and renaming them for better understanding
df_gp = df_gp[['content', 'score', 'at', 'reviewCreatedVersion']]
df_gp.rename(columns={
    'content': 'review',
    'score': 'rating',
    'at': 'date',
    'reviewCreatedVersion': 'app_version'
}, inplace=True)

df_gp.head()


,review,rating,date,app_version
0,it's a good app,3,2026-02-07 08:34:58,0.0.392
1,worst,1,2026-02-07 07:04:14,0.0.392
2,Super and quick though it may work abnormally ...,5,2026-02-07 04:28:29,0.0.392
3,not able to send money to others,1,2026-02-07 00:09:00,NaN
4,Best app.. makes transactions easier,5,2026-02-06 22:08:49,0.0.392


In [7]:
#scrap apple store reviews for the same app

import requests
import pandas as pd

app_id_ios = 1569017982
country_ios = 'ke'

all_reviews_ios = []

url = f"https://itunes.apple.com/{country_ios}/rss/customerreviews/id={app_id_ios}/sortBy=mostRecent/json"
print(f"Fetching App Store reviews from: {url}")

try:
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    data = response.json()

    if 'feed' in data and 'entry' in data['feed']:
        entries = data['feed']['entry']

        # Skip first entry if it's metadata
        if len(entries) > 0 and 'im:rating' not in entries[0]:
            entries = entries[1:]

        for entry in entries:
            review_data = {
                'review': entry.get('content', {}).get('label', None),
                'rating': int(entry.get('im:rating', {}).get('label', 0)),
                'date': entry.get('updated', {}).get('label', None),
                'app_version': entry.get('im:version', {}).get('label', None)
            }
            all_reviews_ios.append(review_data)

    else:
        print("No reviews found.")

except requests.exceptions.RequestException as e:
    print(f"Request failed: {e}")

df_ios = pd.DataFrame(all_reviews_ios)
print(df_ios.head())


Fetching App Store reviews from: https://itunes.apple.com/ke/rss/customerreviews/id=1569017982/sortBy=mostRecent/json
                                              review  rating  \
0                                          Very fast       5   
1  My equity account can’t allow me to send money...       1   
2                                 Real smooth & easy       4   
3  I wanted to send money to another bank only to...       1   
4        Not finishing to launch ..only logo appears       1   

                        date app_version  
0  2026-02-06T10:27:00-07:00       0.4.4  
1  2026-02-06T07:20:57-07:00       0.4.4  
2  2026-02-02T06:55:34-07:00       0.4.4  
3  2026-01-30T01:07:08-07:00       0.4.4  
4  2026-01-27T11:07:41-07:00       0.4.4  


In [8]:
df_ios = df_ios[['review', 'rating', 'date', 'app_version']]
df_ios.head()

,review,rating,date,app_version
0,Very fast,5,2026-02-06T10:27:00-07:00,0.4.4
1,My equity account can’t allow me to send money...,1,2026-02-06T07:20:57-07:00,0.4.4
2,Real smooth & easy,4,2026-02-02T06:55:34-07:00,0.4.4
3,I wanted to send money to another bank only to...,1,2026-01-30T01:07:08-07:00,0.4.4
4,Not finishing to launch ..only logo appears,1,2026-01-27T11:07:41-07:00,0.4.4


In [9]:
df_gp.to_csv("google_play_reviews.csv", index=False)
df_ios.to_csv("app_store_reviews.csv", index=False)

In [10]:
#clean the text

import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    return text

df_gp['clean_review'] = df_gp['review'].apply(clean_text)
df_ios['clean_review'] = df_ios['review'].apply(clean_text)